## Term Analysis

Get features out of the prediction models, and determine what the most republican and democratic terms really are. Look at certain words over time, try to look into changes. First, collect the tables made in previous scripts. 

Imports and setup 

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from paths import RAW, DERIVED, FIGURES

pd.set_option('display.max_rows', 20)

%config InlineBackend.figure_format = 'retina'
plt.rcParams['savefig.dpi'] = 300

wordfigs = FIGURES / 'wordfigs'
wordfigs.mkdir(exist_ok=True)

colors = {'Democrat': '#1f4e9c', 'Republican': '#c8102e'}

In [2]:
overall = pd.read_parquet(DERIVED / 'model_words.parquet')
topics = pd.read_parquet(DERIVED / 'topic_word_coefs.parquet')
overall.shape, topics.shape

((3813205, 6), (1588645, 7))

Most Republican and Most Democratic Words for each congress, and overall. 

In [3]:
top_words = (
    overall
    .groupby(['congress', 'word'])
    .agg(coef=('coef', 'mean'), n_seeds=('seed', 'size'))
    .reset_index()
) # group by word and congress, take averages across seeds. 
top_words = top_words.loc[top_words['n_seeds'] >= 8] # get rid of words that don't show up 


all_congresses = (
    top_words
    .groupby('word')
    .agg(coef=('coef', 'mean'), n_congresses=('congress', 'size'))
    .reset_index()
) # now down to rows, average score across all of them. What words most common? 
all_congresses = all_congresses.loc[all_congresses['n_congresses'] >= 35] 

recent = top_words.loc[top_words['congress'] == 111]
print(all_congresses.shape)
print(recent.shape)

def top_table(words, n):
    dem = words.sort_values('coef').head(n)
    rep = words.sort_values('coef', ascending=False).head(n)
    return pd.DataFrame({
            'Democrat': dem['word'].values,
            'Dem coef': dem['coef'].round(2).values,
            'Republican': rep['word'].values,
            'Rep coef': rep['coef'].round(2).values
        }, index=range(1, n + 1)) # returns a dataframe of the most democrat and most republican words, with count n. 

display(top_table(all_congresses, 15))
display(top_table(recent, 15))

data = [all_congresses,recent]

titles = [
    'All congresses (1873-2011)',
    '111th Congress (2009-11)'
]

fig, axes = plt.subplots(1, 2, figsize=(10, 6)) #two plots, side by side 

for ax, words, title in zip(axes, data, titles):
    dem = words.sort_values('coef').head(15)
    rep = words.sort_values('coef', ascending=False).head(15).sort_values('coef')
    ax.barh(dem['word'], dem['coef'], color=colors['Democrat'], label='Democrat')
    ax.barh(rep['word'], rep['coef'], color=colors['Republican'], label='Republican')
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set(title=title, xlabel='Mean coefficient (+ Rep, − Dem)')
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig.savefig(wordfigs / 'top_party_words.png', bbox_inches='tight')
plt.close(fig)


(3856, 3)
(5022, 4)


,Democrat,Dem coef,Republican,Rep coef
1,cotton,-0.66,spending,0.39
2,people,-0.42,think,0.36
3,sir,-0.34,business,0.26
4,cuts,-0.25,communist,0.25
5,country,-0.24,today,0.20
6,senate,-0.24,thing,0.20
7,workers,-0.23,way,0.19
8,public,-0.22,year,0.18
9,rights,-0.21,wheat,0.17
10,power,-0.20,things,0.17


,Democrat,Dem coef,Republican,Rep coef
1,act,-1.43,government,2.09
2,insurance,-1.18,spending,1.91
3,recovery,-1.09,taxes,1.87
4,work,-0.92,going,1.56
5,support,-0.87,majority,1.29
6,help,-0.87,debt,1.17
7,families,-0.87,stimulus,1.15
8,street,-0.80,trillion,1.07
9,senate,-0.79,thats,0.97
10,important,-0.79,billion,0.90


Most partisan words now, scores over time. 

In [4]:
recent_dem = list(recent.sort_values('coef').head(5)['word'])
recent_rep = list(recent.sort_values('coef', ascending=False).head(5)['word'])

top10_only = top_words.loc[top_words['word'].isin(recent_dem + recent_rep)]

wide = top10_only.pivot(index='congress', columns='word', values='coef') #keeps missing values in the data. 
wide.index = 1789 + 2 * (wide.index - 1) #make index year. for plotting

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True) #one panel per party

for ax, words, party in [(axes[0], recent_dem, 'Democratic'), (axes[1], recent_rep, 'Republican')]:
    wide[words].plot(ax=ax, linewidth=1.2)
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set(
        title=f'Most {party} words in the 111th Congress',
        xlabel='Year',
        ylabel='Coefficient (+ Rep, - Dem)',
    )
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig.savefig(wordfigs / 'polarized_words.png', bbox_inches='tight')
plt.close(fig)



Words by 20-year period. 

In [6]:
periods = (
    top_words
    .assign(period=(top_words['congress'] - 43) // 10 + 1) #numbered periods, 1 to 7. 20 year increments from 1873. 
    .groupby(['period', 'word'])
    .agg(coef=('coef', 'mean'), n_congresses=('congress', 'size'))
    .reset_index()
)
periods = periods.loc[periods['n_congresses'] >= 5] #get rid of noise 
periods_sorted = sorted(periods['period'].unique())

period_years = {}
for period in range(1, 8):
    start = 1873 + (period - 1) * 20
    end = min(start + 19, 2011)
    period_years[period] = f'{start}-{end}'

period_tables = {}
for period in periods_sorted:
    one = periods.loc[periods['period'] == period]
    period_tables[period] = top_table(one, 15) #create tables for most polar words, reusing function from other section
display(period_tables[1])

for period in periods_sorted:
    fig, ax = plt.subplots(figsize=(6, 7))
    one = periods.loc[periods['period'] == period]
    dem = one.sort_values('coef').head(15)
    rep = one.sort_values('coef', ascending=False).head(15).sort_values('coef')

    ax.barh(dem['word'], dem['coef'], color=colors['Democrat'])
    ax.barh(rep['word'], rep['coef'], color=colors['Republican'])

    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set(
        title = f'Period {period} ({period_years[period]})',
        xlabel = 'Mean coefficient (+ Rep, - Dem)',
    )
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)

    fig.savefig(wordfigs / f'top_words_{period}.png', bbox_inches='tight')
    plt.close(fig)


,Democrat,Dem coef,Republican,Rep coef
1,house,-1.10,senate,0.63
2,sir,-0.95,think,0.54
3,people,-0.85,thing,0.47
4,federal,-0.79,nation,0.45
5,government,-0.78,year,0.43
6,power,-0.62,democratic,0.43
7,proper,-0.50,way,0.42
8,taxation,-0.43,course,0.39
9,congress,-0.43,lands,0.38
10,increase,-0.37,undoubtedly,0.38


Polarized Words - words that have been strongly one party vs the other at different times. 

In [16]:
#first, create a df of the words that have flipped parties at any point, going off of period. Swing is the distance, direction is which way. 
flips = (
    periods.sort_values('period').groupby('word')
    .agg(
        most_dem=('coef', 'min'),
        most_rep=('coef', 'max'),
        first=('coef', 'first'),
        last=('coef', 'last'),
        n_periods=('period', 'size'),
    ).reset_index())

flips = flips.loc[
    (flips['n_periods'] >= 4)
    & (flips['most_dem'] <= -0.15)
    & (flips['most_rep'] >= 0.15)]

flips['swing'] = flips['most_rep'] - flips['most_dem']
flips['direction'] = (flips['last'] > flips['first']).map({True: 'Dem to Rep', False: 'Rep to Dem'})

print(flips.sort_values('swing', ascending=False).groupby('direction').head(20).round(2).to_string())



                word  most_dem  most_rep  first  last  n_periods  swing   direction
3126      government     -0.78      0.92  -0.78  0.92          7   1.70  Dem to Rep
5884          senate     -1.02      0.63   0.63 -0.11          7   1.65  Rep to Dem
6457           taxes     -0.23      1.03  -0.16  1.03          7   1.27  Dem to Rep
2815         federal     -0.79      0.47  -0.79  0.47          7   1.27  Dem to Rep
4789          people     -0.96      0.29  -0.85 -0.33          7   1.24  Dem to Rep
425   administration     -0.55      0.54   0.03 -0.32          7   1.09  Rep to Dem
6453             tax     -0.45      0.54  -0.26  0.54          7   1.00  Dem to Rep
1774         country     -0.77      0.17  -0.32 -0.77          7   0.94  Rep to Dem
1                000     -0.27      0.54  -0.27 -0.22          7   0.81  Dem to Rep
937          believe     -0.47      0.33   0.05  0.26          7   0.80  Dem to Rep
1574        congress     -0.43      0.35  -0.43  0.03          7   0.77  Dem

In [18]:
# plots for this, colored by party

dem_to_rep = ['money', 'taxes', 'people', 'war']
rep_to_dem = ['industry', 'work', 'education', 'democratic']

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True, sharey=True)

for ax, word in zip(axes.flat, dem_to_rep + rep_to_dem):
    one = top_words.loc[top_words['word'] == word]
    year = 1789 + 2 * (one['congress'] - 1)
    above = one['coef'] > 0

    ax.plot(year, one['coef'], color='grey', linewidth=0.8)
    ax.scatter(year[above], one.loc[above, 'coef'], color=colors['Republican'], s=12)
    ax.scatter(year[~above], one.loc[~above, 'coef'], color=colors['Democrat'], s=12) # tilde flips the boolean values 
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set(title=word)
    ax.spines[['top', 'right']].set_visible(False)

axes[0, 0].set(ylabel='Coefficient (+ Rep, − Dem)')
axes[1, 0].set(ylabel='Coefficient (+ Rep, − Dem)')

fig.tight_layout()
fig.savefig(wordfigs / 'flip_flop_words.png', bbox_inches='tight')
plt.close(fig)

Topic specific words, Immigration and Abortion

In [ ]:
topic_words = (
    topics
    .groupby(['topic', 'congress', 'word'])
    .agg(coef=('coef', 'mean'), n_seeds=('seed', 'size'))
    .reset_index()
)
topic_words = topic_words.loc[topic_words['n_seeds'] >= 8] #get rid of noise

for topic in ['Immigration', 'Abortion']:
    one = topic_words.loc[topic_words['topic'] == topic]
    n_total = one['congress'].nunique() # how many congresses include this word? 

    words = (
        one
        .groupby('word')
        .agg(coef=('coef', 'mean'), n_congresses=('congress', 'size'))
        .reset_index()
    )
    words = words.loc[words['n_congresses'] >= n_total / 2] # show up in at least half of the congresses with the term
    # already restricted by speech count from topic_party_prediction, only includes years where enough mentions of the word happened 

    display(top_table(words, 15)) # same function

    dem = words.sort_values('coef').head(15)
    rep = words.sort_values('coef', ascending=False).head(15).sort_values('coef')

    first_year = 1789 + 2 * (one['congress'].min() - 1)
    last_year = 1789 + 2 * (one['congress'].max() - 1) + 2 # calcualte the years because different congresses start and end based on whether word was used enough. 

    fig, ax = plt.subplots(figsize=(6, 7))
    ax.barh(dem['word'], dem['coef'], color = colors['Democrat'])
    ax.barh(rep['word'], rep['coef'], color = colors['Republican'])
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set(
        title=f'{topic}: most partisan words ({first_year}-{last_year})',
        xlabel='Mean coefficient (+ Rep, - Dem)',
    )
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)

    fig.savefig(wordfigs / f'{topic.lower()}_top_words.png', bbox_inches='tight')
    plt.close(fig)

,Democrat,Dem coef,Republican,Rep coef
1,families,-0.29,illegal,0.85
2,americans,-0.25,aliens,0.28
3,community,-0.24,think,0.27
4,family,-0.24,alien,0.22
5,country,-0.21,current,0.19
6,city,-0.20,issue,0.19
7,programs,-0.19,government,0.17
8,rights,-0.18,look,0.17
9,children,-0.17,borders,0.16
10,parents,-0.15,control,0.16


,Democrat,Dem coef,Republican,Rep coef
1,women,-1.14,partialbirth,1.09
2,health,-0.54,unborn,0.59
3,reproductive,-0.40,human,0.56
4,womans,-0.40,child,0.52
5,access,-0.38,children,0.49
6,services,-0.38,baby,0.46
7,woman,-0.37,prolife,0.45
8,roe,-0.37,parents,0.45
9,choose,-0.36,demand,0.42
10,funds,-0.36,taxpayers,0.39
